In [ ]:
audio_path =

In [ ]:
import torch
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, PrepareFor kbitTraining

# 1. Load your CSV dataset
dataset = load_dataset("csv", data_files={"train": "metadata.csv"})

# 2. Convert the 'file_name' column into actual audio arrays and resample to 16kHz
dataset = dataset.cast_column("file_name", Audio(sampling_rate=16000))

# 3. Load processor (Feature Extractor + Tokenizer)
model_id = "openai/whisper-small" # Or medium / large-v3
processor = WhisperProcessor.from_pretrained(model_id, language="English", task="transcribe")

# 4. Data Preparation Function
def prepare_dataset(batch):
    audio = batch["file_name"]
    # Extract log-mel spectrogram features
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Encode text transcriptions into token IDs
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names)

In [ ]:
# Load base model
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Define LoRA configuration
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

# Apply LoRA to model
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()